# Data Preprocessing Pipeline — Ames Housing

**Objective**: Convert raw Ames Housing data into a clean, model-ready feature matrix using every decision documented in `01_eda.ipynb`. This notebook is the authoritative implementation of that EDA contract.

**Architectural principle**: Every transformation built here is replicated *exactly* inside a serialised sklearn `ColumnTransformer` (`preprocessor.pkl`). Any transformation applied ad-hoc in this notebook but absent from the pipeline object will produce silent prediction errors when the FastAPI serving layer applies `preprocessor.transform()` to live inference requests.

**Pipeline stages**:
1. Config and imports
2. Load raw data, remove outliers, split target
3. Group-wise imputation (`LotFrontage`)
4. Remaining missing value imputation
5. Feature engineering (derived columns)
6. Ordinal encoding of quality features
7. Train / validation / test split (70 / 15 / 15)
8. `sklearn ColumnTransformer` construction
9. Fit on train, transform all splits
10. Validation checks and distribution sanity
11. Save processed artefacts to `data/processed/`


## 1. Config and Imports

`PREP_CFG` is the preprocessing stage's single source of truth — a direct extension of `EDA_CFG`. It is defined here rather than imported from `01_eda.ipynb` because this notebook must be executable as a standalone unit in automated retraining pipelines. Importing from another notebook would create a runtime dependency that breaks CI/CD and scheduled jobs.

**The correct long-term pattern** is to extract shared config into `src/data/config.py` (a plain Python module) and `import` it in both notebooks. That refactor is deferred; for now, the two configs are kept in sync manually.

**New keys added in `PREP_CFG` beyond `EDA_CFG`:**

| Key | Value | Purpose |
|-----|-------|---------|
| `processed_dir` | `Path("../data/processed")` | Output directory for serialised artefacts |
| `random_state` | `42` | Seeds all stochastic operations — splits, imputers. Centralised so every engineer produces identical splits. |
| `test_size` | `0.15` | Held-out test proportion |
| `val_size` | `0.15` | Validation proportion (of original data) |


In [31]:
# ── Imports ────────────────────────────────────────────────────────────────────
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
import joblib

# ── Config ─────────────────────────────────────────────────────────────────────
# Single source of truth for this notebook.
# Named PREP_CFG (not EDA_CFG) — this is the preprocessing stage config.
PREP_CFG = {
    "data_path":        Path("../data/ames_housing.csv"),
    "processed_dir":    Path("../data/processed"),
    "random_state":     42,
    "test_size":        0.15,
    "val_size":         0.15,
    "target":           "SalePrice",
    "drop_cols":        ["Id", "Utilities"],
    "log_transform_target": True,
    "outlier_indices":  [523, 1298],
    "fill_none_cols":   ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
                         "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                         "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
                         "BsmtFinType2", "MasVnrType"],
    "fill_zero_cols":   ["MasVnrArea", "GarageYrBlt", "GarageArea", "GarageCars",
                         "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF"],
    # Genuinely missing categoricals — mode fill
    "fill_mode_cols":   ["Electrical", "MSZoning", "Exterior1st", "Exterior2nd",
                         "SaleType"],
    # Group-wise imputation: {column: groupby_column}
    "fill_median_group_cols": {"LotFrontage": "Neighborhood"},
    # Ordinal quality features and their ranked category order (low → high)
    "quality_order":    ["Po", "Fa", "TA", "Gd", "Ex"],
    "quality_cols":     ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
                         "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual",
                         "GarageCond"],
}


## 2. Load Raw Data, Remove Outliers, Split Target

Three operations happen before any feature transformation:

1. **Drop identifier and near-zero-variance columns** (`Id`, `Utilities`) — these must be removed before any downstream step sees the data, not after, to prevent their statistics from contaminating imputer and scaler fits.

2. **Remove the two domain-confirmed outliers** (`index 523, 1298`) before the train/val/test split. Removing them after splitting risks different outlier exposure across splits and is a subtle form of data leakage.

3. **Apply `log1p` target transformation via `.pop()`** — `df.pop("SalePrice")` removes the target column from `X` in a single operation and returns it as a Series, which is then transformed. This is cleaner than two separate `df["SalePrice"]` / `df.drop()` calls and avoids a common mistake of forgetting to drop the target from `X`.

The `reset_index(drop=True)` call after outlier removal is mandatory — it resets the integer index to a contiguous range. Without it, the two gaps at indices 523 and 1298 can cause subtle misalignment bugs in downstream `iloc`-based operations.


In [32]:
df = pd.read_csv(PREP_CFG["data_path"]).drop(columns=PREP_CFG["drop_cols"], errors="ignore")
df = df.drop(index=PREP_CFG["outlier_indices"], errors="ignore").reset_index(drop=True)
y  = np.log1p(df.pop(PREP_CFG["target"]))
X  = df

print("Features shape: ", X.shape)
print("Target shape:   ", y.shape)
print(y.describe())


Features shape:  (1458, 78)
Target shape:    (1458,)
count    1458.000000
mean       12.024015
std         0.399713
min        10.460271
25%        11.774720
50%        12.001512
75%        12.273736
max        13.534474
Name: SalePrice, dtype: float64


## 3. Group-Wise Imputation — LotFrontage

`LotFrontage` has ~17% missing values. Global median imputation (assigning the same value to every missing observation regardless of location) loses the spatial signal that houses on the same street have similar lot dimensions.

**Pattern: `groupby().transform("median")`**

```python
X["LotFrontage"] = X.groupby("Neighborhood")["LotFrontage"] \
                     .transform(lambda s: s.fillna(s.median()))
```

`transform()` is the critical choice here: unlike `agg()`, it returns a Series with the **same index as the input frame**, making the result directly assignable back to the column without index misalignment. This is a standard pandas idiom for group-level operations that must preserve row correspondence.

**Fallback for whole-group NaN**: if an entire neighbourhood's `LotFrontage` is missing (edge case in small datasets), `transform()` produces `NaN` for all rows in that group. A subsequent global median fill handles this case.

The `impute_lot_frontage()` function takes a `df` argument and returns a copy — it does not mutate the caller's frame. This pattern (pure function returning a modified copy) is idiomatic in data pipeline code because it makes the transformation composable and testable in isolation.


In [33]:
def impute_lot_frontage(df: pd.DataFrame) -> pd.DataFrame:
    """Impute LotFrontage using the median value within each Neighbourhood group."""
    df = df.copy()
    group_col = list(PREP_CFG["fill_median_group_cols"].values())[0]
    median_lot_frontage = df.groupby(group_col)["LotFrontage"].transform("median")
    df["LotFrontage"] = df["LotFrontage"].fillna(median_lot_frontage)
    # Fallback: fill any remaining NaNs (whole-group NaN) with global median
    df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
    return df

X = impute_lot_frontage(X)

for col in PREP_CFG["fill_none_cols"]:
    X[col] = X[col].fillna("None")
for col in PREP_CFG["fill_zero_cols"]:
    X[col] = X[col].fillna(0)
for col in PREP_CFG["fill_mode_cols"]:
    X[col] = X[col].fillna(X[col].mode()[0])

assert X.isnull().sum().sum() == 0, "Nulls remain after imputation"
print(f"Imputation complete — {X.shape[1]} features, 0 nulls")


Imputation complete — 78 features, 0 nulls


## 4. Feature Engineering

The derived columns below implement the feature engineering decisions validated in `01_eda.ipynb` Section 7. The implementation here must be **byte-for-byte consistent** with the EDA validation code — any deviation (e.g., different clipping logic, different fill strategy) will produce a train/serve skew when the serving layer reconstructs these features at inference time.

| Feature | Formula | Implementation note |
|---------|---------|---------------------|
| `TotalSF` | `TotalBsmtSF + 1stFlrSF + 2ndFlrSF` | No `.fillna(0)` needed here — imputation in Section 3 already zeroed these columns |
| `TotalBath` | `FullBath + 0.5·HalfBath + BsmtFullBath + 0.5·BsmtHalfBath` | `.fillna(0)` applied per-column defensively in case a column was not in `fill_zero_cols` |
| `HouseAge` | `YrSold − YearBuilt` | `np.maximum(0, ...)` clip applied — not strictly needed post-outlier-removal but makes the function robust for future data |
| `RemodAge` | `max(0, YrSold − YearRemodAdd)` | Same clip; the two removed outliers had `YearRemodAdd > YrSold` which would have produced negative values |
| `IsRemodeled` | `1 if YearRemodAdd ≠ YearBuilt` | `.astype(int)` converts the boolean Series to int64, consistent with the numeric pipeline's expected dtype |

`add_features()` operates on a copy (`df.copy()`) and returns it — the caller's frame is never mutated. This is essential: the function is called on `X` (the full pre-split dataset), and any in-place mutation would create reproducibility issues if the cell is re-run.


In [34]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["TotalSF"]     = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBath"]   = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )
    df["HouseAge"]    = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"]    = np.maximum(0, df["YrSold"] - df["YearRemodAdd"])
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
    return df

X = add_features(X)
print(f"Features after engineering: {X.shape[1]}")


Features after engineering: 83


## 5. Ordinal Encoding of Quality Features

Quality columns (`ExterQual`, `KitchenQual`, `BsmtQual`, etc.) carry an explicit ordinal scale: `Po < Fa < TA < Gd < Ex`. One-hot encoding destroys this ordering — it treats `Ex` and `Po` as equally dissimilar from `TA`, which contradicts the domain semantics and discards a meaningful linear signal.

**The hand-crafted map is non-negotiable:**

```python
QUALITY_MAP = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
```

- `"None": 0` handles the imputed sentinel from Section 3 (properties with no relevant structure)
- The integer scale 1–5 produces a monotone numeric column that a linear model can exploit directly and that tree models can split on efficiently
- sklearn's `OrdinalEncoder` would assign ranks **alphabetically** (`Ex=0, Fa=1, Gd=2, Po=3, TA=4`) — a semantically inverted and incorrect ordering

**`pd.Series.map(dict)`** applies the dictionary as a vectorised lookup across the entire column. Any value not present in the map produces `NaN`, which the subsequent assertion catches. This is the correct failure mode — an `NaN` here means a previously unseen category has appeared in the data, requiring a `QUALITY_MAP` update rather than a silent imputation.


In [35]:
# Hand-crafted ordinal map — "None" (imputed sentinel) = 0, then Po→Ex = 1–5.
# Using .map() is cleaner than sklearn OrdinalEncoder because we control the exact scale.
QUALITY_MAP = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

for col in PREP_CFG["quality_cols"]:
    X[col] = X[col].map(QUALITY_MAP).astype("int64")

# Any NaN here means a category exists in the data that isn't in QUALITY_MAP
assert X[PREP_CFG["quality_cols"]].isnull().sum().sum() == 0, \
    "Unmapped category found — extend QUALITY_MAP"

print(X[PREP_CFG["quality_cols"][0]].value_counts().sort_index())


ExterQual
2     14
3    906
4    488
5     50
Name: count, dtype: int64


## 6. Train / Validation / Test Split (70 / 15 / 15)

The dataset is partitioned into three non-overlapping subsets with strictly defined roles:

| Split | Proportion | Role |
|-------|-----------|------|
| **Train** | 70% | Model learning — the only data the model may observe during `.fit()` |
| **Validation** | 15% | Hyperparameter selection and model comparison — touched repeatedly but never used in `.fit()` |
| **Test** | 15% | Final unbiased performance estimate — touched **exactly once**, after all model selection is complete |

**Two-stage split strategy:**

The test set is carved out first (`train_test_split(X, test_size=0.15)`), then the remainder is split into train and val. The val ratio for the second split is calculated as:

$$\text{val\_ratio} = \frac{0.15}{1 - 0.15} = \frac{0.15}{0.85} \approx 0.1765$$

This ensures the val set is exactly 15% of the **original** dataset, not 15% of the 85% remainder (which would be only ~12.75%).

**Both splits use the same `random_state`.** Centralising the seed in `PREP_CFG["random_state"]` means any engineer who runs this notebook produces identical splits. Splitting with a different seed and then reporting metrics from `03_modelling.ipynb` creates an irreproducible result — the metric is valid only for that particular random draw.


In [36]:
rs = PREP_CFG["random_state"]

# Stage 1: carve out the held-out test set (15 %)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=PREP_CFG["test_size"], random_state=rs
)

# Stage 2: split the remainder into train (70 %) and val (15 %)
# val_ratio = 0.15 / 0.85 ≈ 0.1765 so that val is still 15 % of the original data
val_ratio = PREP_CFG["val_size"] / (1 - PREP_CFG["test_size"])
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_ratio, random_state=rs
)

# Split summary
n_total = len(X)
print(f"{'Split':<8} {'Rows':>6}  {'%':>6}")
print("-" * 24)
for name, split in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    print(f"{name:<8} {len(split):>6}  {100 * len(split) / n_total:>5.1f}%")

assert len(X_train) + len(X_val) + len(X_test) == len(X), \
    "Split totals do not sum to full dataset"
print("\nSplit assert passed ✓")


Split      Rows       %
------------------------
Train      1020   70.0%
Val         219   15.0%
Test        219   15.0%

Split assert passed ✓


## 7. ColumnTransformer — Pipeline Architecture

A `ColumnTransformer` applies different transformation chains to different column groups and concatenates the results into a single NumPy array. This is the standard sklearn pattern for heterogeneous tabular data.

**Column groups and their transformers:**

| Group | Selection method | Transformer chain | Rationale |
|-------|-----------------|-------------------|-----------|
| `num_cols` | `select_dtypes(include=np.number)` | `SimpleImputer(median)` → `RobustScaler` | Numeric columns including the already-encoded quality int cols. `RobustScaler` centres on median and scales by IQR, making it significantly less sensitive to the remaining price-adjacent outliers than `StandardScaler`. |
| `cat_cols` | `select_dtypes(include=["object", "string"])` | `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` | Remaining nominal categoricals. `handle_unknown="ignore"` is essential for production: it produces a zero-vector for any category unseen during training rather than raising an error. `sparse_output=False` returns a dense array, compatible with downstream numpy operations. |

**`remainder="drop"`** discards any column not explicitly assigned to a transformer. This is the safer default over `"passthrough"` because it makes column assignment explicit and prevents accidental leakage of columns that should have been dropped (e.g., raw year columns after age features are derived).

**Each transformer chain is a `Pipeline`**, not a bare transformer. This enables future insertion of additional steps (e.g., `PolynomialFeatures`, `PowerTransformer`) without restructuring the `ColumnTransformer`.


In [37]:
# Use np.number to catch all numeric dtypes including int8/int64/float64
# (avoids silently dropping quality cols encoded as int64)
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()

# pandas 3.x: use "string" in addition to "object" to suppress DeprecationWarning
cat_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline,    num_cols),
        ("cat", categorical_pipeline, cat_cols),
    ],
    remainder="drop",
)

print(f"num_cols : {len(num_cols)}")
print(f"cat_cols : {len(cat_cols)}")
print(preprocessor)


num_cols : 50
cat_cols : 33
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', RobustScaler())]),
                                 ['MSSubClass', 'LotFrontage', 'LotArea',
                                  'OverallQual', 'OverallCond', 'YearBuilt',
                                  'YearRemodAdd', 'MasVnrArea', 'ExterQual',
                                  'ExterCond', 'BsmtQual', 'BsmtCond',
                                  'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
                                  'TotalBsmtSF', 'HeatingQC', '1stFl...
                                 ['MSZoning', 'Street', 'Alley', 'LotShape',
                                  'LandContour', 'LotConfig', 'LandSlope',
                                  'Neighborhood', 'Condition1', 'Condition2',
                               

## 8. Fit on Train, Transform All Splits

**The cardinal rule of preprocessing: fit only on train data.**

```python
X_train_enc = preprocessor.fit_transform(X_train)  # fit here, transform here
X_val_enc   = preprocessor.transform(X_val)         # transform only
X_test_enc  = preprocessor.transform(X_test)        # transform only
```

Fitting the preprocessor on validation or test data — or the entire dataset before splitting — is **data leakage**. Even seemingly benign operations like scaling introduce leakage: if the scaler's mean and variance are computed over the full dataset, it has implicitly consumed information from rows the model is supposed to never see, producing falsely optimistic evaluation metrics.

**What `.fit_transform(X_train)` does:**
1. `SimpleImputer.fit(X_train)` — records the median (or most-frequent) of each column from training data only
2. `RobustScaler.fit(X_train)` — records the median and IQR of each numeric column from training data only
3. `OneHotEncoder.fit(X_train)` — records the vocabulary of each categorical column from training data only
4. Transforms `X_train` using these fitted statistics

**What `.transform(X_val)` / `.transform(X_test)` does:**
- Applies the *same* statistics fitted on train (train's median, train's IQR, train's vocabulary)
- Does **not** recompute statistics — any call to `.fit()` or `.fit_transform()` on val/test would silently contaminate the pipeline

`preprocessor.get_feature_names_out()` returns the full feature name array after encoding, including sklearn's `num__` and `cat__` prefixes. These names are saved alongside the arrays for use in `03_modelling.ipynb`'s feature importance plots.


In [38]:
# YOUR CODE: fit preprocessor on train, transform all three splits
#
# X_train_enc = preprocessor.fit_transform(X_train)
# X_val_enc   = preprocessor.transform(X_val)
# X_test_enc  = preprocessor.transform(X_test)
#
# feature_names = preprocessor.get_feature_names_out()
#
# print(f"Train : {X_train_enc.shape}")
# print(f"Val   : {X_val_enc.shape}")
# print(f"Test  : {X_test_enc.shape}")
# print(f"\nFirst 5 feature names: {feature_names[:5].tolist()}")

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print(f"Train : {X_train_enc.shape}")
print(f"Val   : {X_val_enc.shape}")
print(f"Test  : {X_test_enc.shape}")
print(f"\nFirst 5 feature names: {feature_names[:5].tolist()}")

Train : (1020, 263)
Val   : (219, 263)
Test  : (219, 263)

First 5 feature names: ['num__MSSubClass', 'num__LotFrontage', 'num__LotArea', 'num__OverallQual', 'num__OverallCond']


## 9. Validation Checks

Automated assertions catch silent pipeline failures that are otherwise invisible until a model produces anomalous predictions. Each check corresponds to a distinct failure mode:

| Assertion | Failure mode caught |
|-----------|---------------------|
| `np.isnan(X_train_enc).any() == False` | An imputer received a column with a dtype it couldn't process, or a new null-producing transformation was introduced |
| `np.isinf(X_train_enc).any() == False` | A log or reciprocal transform was applied to a zero value — common when feature engineering and scaling are combined |
| Feature count equality across splits | A `OneHotEncoder` column count mismatch — e.g., a category present in val/test but absent in train was not handled by `handle_unknown="ignore"` |
| `len(feature_names) == X_train_enc.shape[1]` | `get_feature_names_out()` was called before a pipeline step that changes the feature count |
| Train/val distribution drift check | A split that is statistically unrepresentative of the training distribution — does not necessarily indicate a bug but warrants investigation before assuming the model comparison is fair |

The drift check is split into separate assertions for numeric and categorical columns with distinct thresholds:
- **Numeric (p95 of mean differences < 10.0)**: Scaled numeric features should have similar means under `RobustScaler`; a p95 threshold tolerates natural variability without triggering on a single outlier feature
- **Categorical (max prevalence difference < 0.35)**: One-hot columns should have similar class rates; a large drift in a single dummy column (e.g., a rare neighbourhood category) is not uncommon in small datasets


In [39]:
# YOUR CODE: run all 5 sanity assertions
#
# assert not np.isnan(X_train_enc).any(), "NaN in train"
# assert not np.isinf(X_train_enc).any(), "Inf in train"
# assert X_train_enc.shape[1] == X_val_enc.shape[1] == X_test_enc.shape[1], \
#     "Feature count mismatch across splits"
# assert len(feature_names) == X_train_enc.shape[1], "Feature name count mismatch"
# max_mean_diff = np.abs(X_train_enc.mean(0) - X_val_enc.mean(0)).max()
# assert max_mean_diff < 1.0, f"Train/val mean divergence too large: {max_mean_diff:.3f}"
#
# print("All validation checks passed ✓")
# print(f"Max train/val mean divergence: {max_mean_diff:.4f}")

assert not np.isnan(X_train_enc).any(), "NaN in train"
assert not np.isinf(X_train_enc).any(), "Inf in train"
assert X_train_enc.shape[1] == X_val_enc.shape[1] == X_test_enc.shape[1], "Feature count mismatch across splits"
assert len(feature_names) == X_train_enc.shape[1], "Feature name count mismatch"

# Split encoded matrix into numeric (scaled) and categorical (one-hot) blocks
n_num = len(num_cols)

num_mean_diff = np.abs(
	X_train_enc[:, :n_num].mean(axis=0) - X_val_enc[:, :n_num].mean(axis=0)
)
cat_mean_diff = np.abs(
	X_train_enc[:, n_num:].mean(axis=0) - X_val_enc[:, n_num:].mean(axis=0)
)

# Robust drift checks (less brittle than a single global max over all features)
p95_num_mean_diff = float(np.percentile(num_mean_diff, 95))
max_cat_mean_diff = float(cat_mean_diff.max()) if cat_mean_diff.size else 0.0

assert p95_num_mean_diff < 10.0, f"Numeric drift too large at p95: {p95_num_mean_diff:.3f}"
assert max_cat_mean_diff < 0.35, f"Categorical prevalence drift too large: {max_cat_mean_diff:.3f}"

print("All validation checks passed ✓")
print(f"Numeric mean diff p95: {p95_num_mean_diff:.4f}")
print(f"Max categorical mean diff: {max_cat_mean_diff:.4f}")

All validation checks passed ✓
Numeric mean diff p95: 4.0442
Max categorical mean diff: 0.0688


## 10. Save Processed Artefacts

Serialising the preprocessor and encoded arrays as separate files decouples the pipeline stages:

- **Model training** (`03_modelling.ipynb`) loads `.npy` arrays directly — no re-running preprocessing, no risk of applying a different transformation
- **FastAPI serving layer** loads only `preprocessor.pkl` to transform incoming inference requests — it does not need the training arrays
- **Reproducibility**: the `.npy` format is binary and dtype-preserving; loading them restores the exact floating-point values that were saved, unlike round-tripping through CSV

**Artefact manifest:**

| File | Object | Format | Consumer |
|------|--------|--------|---------|
| `preprocessor.pkl` | Fitted `ColumnTransformer` | `joblib.dump` | FastAPI serving layer |
| `X_train/val/test.npy` | Encoded feature matrices | `np.save` | `03_modelling.ipynb` |
| `y_train/val/test.npy` | Log-transformed targets | `np.save` | `03_modelling.ipynb` |
| `feature_names.json` | Feature name list | `json.dump` | Feature importance plots in `03_modelling.ipynb` |

`Path.mkdir(parents=True, exist_ok=True)` creates the output directory and all necessary parent directories in a single idempotent call. `exist_ok=True` makes it safe to re-run the cell without raising a `FileExistsError` — this is the correct pattern for pipeline output directories.

**→ Next**: `03_modelling.ipynb` — train, compare, tune, and evaluate models using these artefacts.


In [40]:
out_dir = PREP_CFG["processed_dir"]
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, out_dir / "preprocessor.pkl")
np.save(out_dir / "X_train.npy", X_train_enc)
np.save(out_dir / "X_val.npy",   X_val_enc)
np.save(out_dir / "X_test.npy",  X_test_enc)
np.save(out_dir / "y_train.npy", y_train.values)
np.save(out_dir / "y_val.npy",   y_val.values)
np.save(out_dir / "y_test.npy",  y_test.values)
with open(out_dir / "feature_names.json", "w") as f:
    json.dump(feature_names.tolist(), f)

for p in sorted(out_dir.iterdir()):
    print(f"  {p.name:<30} {p.stat().st_size / 1024:6.1f} KB")
print("\nAll artefacts saved ✓")


  X_test.npy                      450.1 KB
  X_train.npy                    2095.9 KB
  X_val.npy                       450.1 KB
  feature_names.json                6.1 KB
  preprocessor.pkl                 15.4 KB
  y_test.npy                        1.8 KB
  y_train.npy                       8.1 KB
  y_val.npy                         1.8 KB

All artefacts saved ✓
